In [0]:
# Type casting is not necessary here as we already have to correct data types, as per our previous result of the sql query in the bronze layer code

In [0]:
#Now let's perform some data anonymization tasks, data validation tasks
from pyspark.sql import functions as F
bronze_orders_df=spark.read.table("db_retail_analytics.supply_chain_project.bronze_orders")

silver_orders_df=(
    bronze_orders_df
    #Masking the cusomer_id, this is already done according to the original dataset, but we still do again to get the concept of ananomizaiton
    .withColumn("customer_id",F.sha2(F.col("customer_id"),256))

    # checking the data quality by flagging the bad columns(data integrity)
    .withColumn(
        "is_valid_timeline",
        F.when(
            (F.col("order_delivered_customer_date").isNotNull()) & 
            (F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp")), 
            False
        ).otherwise(True)
    )

)
# saving this silver tables
(
    silver_orders_df.
    write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_orders")
)

print("The silver table is saved/overwritten")

In [0]:
%sql select * from db_retail_analytics.supply_chain_project.bronze_order_items LIMIT 5;

In [0]:
# silver layer transformations for order_items
order_items_bronze_df=(
    spark.
    read.
    table("db_retail_analytics.supply_chain_project.bronze_order_items")
)

silver_order_items_df=(
    order_items_bronze_df
    #making sure our primary key row doesn't have null in it
    .filter(F.col("order_id").isNotNull())
    #data hygeine 
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    # we need to cast but as from the above sql result we can see we don't need to cast anything because all the column we need are in the same datatype
    #data validation: keeping only the rows with price>0 as they can make bad values
    .filter(F.col("price")>0)
)
# saving our table in the Silver layer
(
    silver_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_order_items")
)

print("silver order items table save successfully")

In [0]:
'''
From this cell the code is done after the Minimum Viable Pipeline (initial vertical slice of the architecture(2 kpis created in powerbi))
'''

In [0]:
'''
As per our metrics we can understand 90% of the orders are on time, but we are going to see where the 10% of the orders are delayed
'''


In [0]:
%%sql
SELECT * FROM db_retail_analytics.supply_chain_project.bronze_geolocation LIMIT 5;

In [0]:
'''
rather than just dropping the zip_code_prefix, let's do the aggregation because for powerbi if we have the average of lat, long the map will be more interesting with locaitons pointing to the middle of the zipcode
'''
from pyspark.sql import functions as F
bronze_geo=spark.read.table("db_retail_analytics.supply_chain_project.bronze_geolocation")

silver_geo=(
    bronze_geo
    .filter(F.col("geolocation_zip_code_prefix").isNotNull())
    .groupBy(F.col("geolocation_zip_code_prefix"))
    .agg(
        F.avg(F.col("geolocation_lat")).alias("latitude"),
        F.avg(F.col("geolocation_lng")).alias("longitude"),
        #now let's remove the trailing and ending space in the text columns
        F.first(F.trim(F.col("geolocation_city"))).alias("city"),
        F.first(F.trim(F.col("geolocation_state"))).alias("state")
    )
)

#it's time we have to save this into a table

(
    silver_geo
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_geolocaiton")
)

In [0]:
'''
Now we are done with the bronze table of customers we will clean so the briging would be better to the gold tables
'''


In [0]:
from pyspark.sql import functions as F
customers_bronze_df=(
    spark.
    read.
    table("db_retail_analytics.supply_chain_project.bronze_customers")
)

silver_customers_df=(
    customers_bronze_df
    .filter(F.col("customer_id").isNotNull())
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_unique_id",F.trim(F.col("customer_unique_id")))
    .withColumn("customer_city", F.trim(F.col("customer_city")))
    .withColumn("customer_state",F.trim(F.col("customer_state")))
    .dropDuplicates(["customer_id"])
    #as we did earlier hasing the customer ID(anonymization) we are going to do the same thing here, but don't forget to use the same algorithm we used earlier, because customer ID using same alogithm as earlier gives same values as we got earlier, if not we loose the data
    .withColumn("customer_id",F.sha2(F.col("customer_id"),256))
    # The reason for select statement is to create an alias for the column names, and for the zip_code I don't change the name because i wanted to explicitly show how I join/broadcast with different names in the gold table 
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        F.col("customer_city").alias("city"),
        F.col("customer_state").alias("state")
    )
)

(
    silver_customers_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_customers")
)

In [0]:
%%sql
SELECT * FROM db_retail_analytics.supply_chain_project.bronze_customers LIMIT 5;